# Croppy: build the offline region pack
This builds one small JSON file (about 5 KB) with everything the app needs offline for one area: recent weather, climate normals, soil, crop calendar, national yields and a fair-price band. Load it into the app under **My farm → Load region pack**, or copy it to `app/region/pack.json`.

Rebuild it whenever new prices arrive. At about 5 KB it fits in one cheap data bundle (store-and-forward).

| Part | Source | How |
|---|---|---|
| Recent weather + normals | NASA POWER (no key needed) | API, by coordinates |
| Soil pH, nitrogen, organic carbon | ISRIC SoilGrids (or iSDAsoil for Africa, 30 m) | API, by coordinates |
| Maize and bean prices | WFP food prices (HDX CSV for your country) | Upload the CSV |
| Coffee farm-gate price | National coffee board or co-op price sheet | Type in the monthly values |
| National yields | FAOSTAT (QCL domain) | Type in or upload |
| Harvest months | FAO GIEWS crop calendar + extension officer | Type in |

In [ ]:
import requests, json, datetime as dt, statistics as st, pandas as pd, numpy as np
# ---- EDIT THESE ----
REGION = {"en":"Mt Elgon, Bugisu sub-region, Uganda","sw":"Mlima Elgon, Bugisu, Uganda"}
LAT, LON = 1.15, 34.35          # centre of the farm area
CURRENCY = "UGX"
TODAY = dt.date.today()
SOURCES = []

## 1. Weather: last 60 days and 30-year monthly normals (NASA POWER)

In [ ]:
P="T2M_MIN,T2M_MAX,PRECTOTCORR,RH2M"
start=(TODAY-dt.timedelta(days=60)).strftime("%Y%m%d"); end=TODAY.strftime("%Y%m%d")
r=requests.get(f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters={P}&community=AG&latitude={LAT}&longitude={LON}&start={start}&end={end}&format=JSON",timeout=60).json()
p=r["properties"]["parameter"]; recent=[]
for d in sorted(p["T2M_MIN"]):
    row={"date":f"{d[:4]}-{d[4:6]}-{d[6:]}","tmin":p["T2M_MIN"][d],"tmax":p["T2M_MAX"][d],"rain":p["PRECTOTCORR"][d],"rh":p["RH2M"][d]}
    if min(v for k,v in row.items() if k!="date")>-900: recent.append({k:(round(v,1) if k!="date" else v) for k,v in row.items()})
print(len(recent),"days, last:",recent[-1]["date"])   # POWER lags a few days behind today

c=requests.get(f"https://power.larc.nasa.gov/api/temporal/climatology/point?parameters={P}&community=AG&latitude={LAT}&longitude={LON}&format=JSON",timeout=60).json()["properties"]["parameter"]
M=["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]; DAYS=[31,28,31,30,31,30,31,31,30,31,30,31]
normals=[{"m":i+1,"tmin":round(c["T2M_MIN"][m],1),"tmax":round(c["T2M_MAX"][m],1),"rain":round(c["PRECTOTCORR"][m]*DAYS[i])} for i,m in enumerate(M)]
print(pd.DataFrame(normals))
SOURCES.append(f"NASA POWER daily and climatology, {LAT},{LON}, retrieved {TODAY}")

## 2. Soil (SoilGrids, 0–15 cm). If the service is down, enter values from iSDAsoil by hand.

In [ ]:
soil={"depth":"0-15 cm"}
try:
    q=requests.get("https://rest.isric.org/soilgrids/v2.0/properties/query",params={"lon":LON,"lat":LAT,"property":["phh2o","nitrogen","soc"],"depth":["0-5cm","5-15cm"],"value":"mean"},timeout=60).json()
    vals={}
    for L in q["properties"]["layers"]:
        v=[d["values"]["mean"] for d in L["depths"] if d["values"]["mean"] is not None]
        vals[L["name"]]=sum(v)/len(v) if v else None
    soil.update({"ph":round(vals["phh2o"]/10,1),"n_gkg":round(vals["nitrogen"]/100,2),"soc_gkg":round(vals["soc"]/10,1)})  # SoilGrids units: pH*10, cg/kg, dg/kg
    soil["source"]=f"ISRIC SoilGrids 2.0, 0-15 cm mean, retrieved {TODAY}"
except Exception as e:
    print("SoilGrids failed:",e); soil.update({"ph":None,"n_gkg":None,"soc_gkg":None,"source":"enter manually from iSDAsoil"})
print(soil); SOURCES.append(soil["source"])

## 3. Prices
**Maize and beans:** download your country's *WFP food prices* CSV from HDX and upload it. The cell uses the markets nearest to your farm and the last 3 months.

**Coffee:** type in the last 6 monthly farm-gate prices for parchment, from the national coffee board or your co-op. Never invent these; write down where each came from.

In [ ]:
from google.colab import files
up=files.upload(); wfp=pd.read_csv(list(up)[0],skiprows=[1])        # HDX files have a #tag row under the header
wfp["date"]=pd.to_datetime(wfp["date"]); wfp["price"]=pd.to_numeric(wfp["price"],errors="coerce")
wfp=wfp[wfp.currency==CURRENCY]
wfp["dist"]=np.hypot(pd.to_numeric(wfp.latitude)-LAT,pd.to_numeric(wfp.longitude)-LON)
def band(pattern, label, crop, prefer="Wholesale"):
    d=wfp[wfp.commodity.str.contains(pattern,case=False)]
    if prefer in set(d.pricetype): d=d[d.pricetype==prefer]
    d=d[d.unit.str.lower().str.strip()=="kg"]
    near=d.drop_duplicates("market").nsmallest(3,"dist").market.tolist(); d=d[d.market.isin(near)]
    last=d.date.max(); recent=d[d.date>last-pd.DateOffset(months=3)].price
    trend=d.groupby(d.date.dt.to_period("M")).price.median().tail(6).round().tolist()
    return {"crop":crop,"label":label,"unit":"kg","low":round(recent.quantile(.25)),"high":round(recent.quantile(.75)),"ref":round(recent.median()),
            "trend":trend,"date":last.strftime("%Y-%m"),"source":f"WFP food prices via HDX ({prefer}), markets: {', '.join(near)}"}
prices={"maize":band("maize","",'maize'),"beans":band("beans","","beans")}
prices["maize"]["label"]={"en":"Maize grain","sw":"Mahindi"}; prices["beans"]["label"]={"en":"Beans","sw":"Maharagwe"}

# ---- Coffee farm-gate prices: EDIT with real monthly values (oldest → newest) ----
COFFEE_MONTHLY = [None,None,None,None,None,None]    # e.g. [8600, 8900, ...] per kg parchment
COFFEE_SOURCE  = "Uganda Coffee Development Authority farm-gate price, Mbale"   # where each number came from
COFFEE_MONTH   = "2026-09"
if all(COFFEE_MONTHLY):
    last3=COFFEE_MONTHLY[-3:]
    prices["coffee_parchment"]={"crop":"coffee","label":{"en":"Coffee parchment","sw":"Kahawa ya parchment"},"unit":"kg",
        "low":round(min(last3)*0.97),"high":round(max(last3)*1.03),"ref":round(st.median(last3)),"trend":COFFEE_MONTHLY,"date":COFFEE_MONTH,"source":COFFEE_SOURCE}
else: print("⚠ Coffee prices not entered: coffee will be missing from Sell.")
for k,v in prices.items(): print(k,v["low"],"-",v["high"],v["date"]); SOURCES.append(v["source"])

## 4. Crops: harvest months and national yields (check with the extension officer and FAOSTAT)

In [ ]:
crops={
 "coffee":{"label":{"en":"Coffee","sw":"Kahawa"},"harvest_months":[10,11,12],"national_yield_kg_ha":None,"yield_source":"FAOSTAT QCL, coffee green, <country>, <year>"},
 "maize": {"label":{"en":"Maize","sw":"Mahindi"},"harvest_months":[7,8,12,1],"national_yield_kg_ha":None,"yield_source":"FAOSTAT QCL, maize, <country>, <year>"},
 "beans": {"label":{"en":"Beans","sw":"Maharagwe"},"harvest_months":[6,7,11,12],"national_yield_kg_ha":None,"yield_source":"FAOSTAT QCL, beans dry, <country>, <year>"}}
# FAOSTAT reports yield in kg/ha in recent releases (older: hg/ha, divide by 10). Fill in the numbers above.

## 5. Write and download the pack

In [ ]:
pack={"synthetic":False,"region":REGION,"built":TODAY.isoformat(),"location":{"lat":LAT,"lon":LON},"currency":CURRENCY,
      "climate":{"normals":normals,"recent":recent[-30:],"source":"NASA POWER"},"soil":soil,"crops":crops,"prices":prices,"sources":SOURCES}
s=json.dumps(pack,separators=(",",":"),ensure_ascii=False); open("pack.json","w").write(s)
print(len(s.encode()),"bytes"); files.download("pack.json")